# Semantic Search & LLM Application
**Starter Notebook (dikerjakan)**

Notebook ini adalah prototype aplikasi semantic search yang terhubung ke LLM, dengan knowledge base FAQ e-commerce.

Alurnya: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

| Tahap | Section |
|---|---|
| **Ingest** | 1–2 |
| **Embed** | 3 |
| **Store** | 4 |
| **Retrieve** | 5 |
| **Generate** | 6–7 |
| **Evaluate** | 8 |

> **Sebelum menjalankan:** buat file `.env` berisi `GEMINI_API_KEY=...` di root project (lihat `.env.example`), lalu jalankan *Kernel → Restart & Run All*.

## 1. Import Library & Load Knowledge Base

In [3]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from google import genai
from google.genai import types

# ---------------- Konfigurasi ----------------
# Nama model ditaruh di satu tempat supaya mudah diganti kalau ada model baru / deprecation.
EMBEDDING_MODEL = "gemini-embedding-2"   # model embedding Gemini terbaru (GA April 2026)
EMBEDDING_DIM = 768                        # 768 cukup untuk 59 FAQ, lebih hemat dari default 3072
# Catatan: text-embedding-004 & gemini-2.0-flash (dipakai starter notebook) sudah dimatikan Google di 2026.
LLM_MODEL = "gemini-3.5-flash-lite"       # utama: cepat, murah, cukup untuk menjawab dari context FAQ
LLM_FALLBACK_MODEL = "gemini-3.1-flash-lite"  # cadangan bila model utama error/overload/kuota habis
TOP_K = 3

# ---------------- Path ----------------
# Notebook bisa dijalankan dari folder notebook/ maupun dari root project.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "ecommerce_faq_knowledge_base.csv"
VECTOR_DIR = PROJECT_ROOT / "vector_store"
DOC_DIR = PROJECT_ROOT / "documentation"
VECTOR_DIR.mkdir(exist_ok=True)
DOC_DIR.mkdir(exist_ok=True)

# ---------------- API key ----------------
# API key dibaca dari file .env (tidak pernah ditulis di kode / di-commit ke GitHub).
load_dotenv(find_dotenv(usecwd=True))
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Create a .env file in the project root (see .env.example).")

# - timeout 60 detik per request: kalau server lambat/macet, request digagalkan (tidak menggantung).
# - retry bawaan SDK dimatikan (attempts=1) karena retry & fallback kita tangani sendiri secara
#   eksplisit di bawah, supaya tidak terjadi retry berlapis yang membuat notebook terlihat "hang".
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=60_000, retry_options=types.HttpRetryOptions(attempts=1)),
)

df = pd.read_csv(DATA_PATH)
print("Total entri knowledge base:", len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


In [4]:
# Periksa struktur data: kolom, tipe data, nilai kosong, duplikat
print("Shape            :", df.shape)
print("Kolom            :", list(df.columns))
print("Nilai kosong     :", int(df.isna().sum().sum()))
print("faq_id duplikat  :", int(df["faq_id"].duplicated().sum()))
print("Question duplikat:", int(df["question"].duplicated().sum()))
print()
print(df.dtypes)

Shape            : (59, 4)
Kolom            : ['faq_id', 'category', 'question', 'answer']
Nilai kosong     : 0
faq_id duplikat  : 0
Question duplikat: 0

faq_id      int64
category      str
question      str
answer        str
dtype: object


In [6]:
# Distribusi kategori dan panjang teks
print("Jumlah kategori:", df["category"].nunique())
display(df["category"].value_counts().rename("jumlah_faq").to_frame())

length_stats = pd.DataFrame({
    "question_chars": df["question"].str.len(),
    "answer_chars": df["answer"].str.len(),
}).describe().round(1)
length_stats

Jumlah kategori: 8


,jumlah_faq
category,
Pemesanan,9
Pengiriman,9
Pembayaran,8
Pengembalian,8
Akun,7
Produk,7
Promo,6
Customer Service,5


,question_chars,answer_chars
count,59.0,59.0
mean,55.0,128.1
std,10.0,17.4
min,35.0,88.0
25%,46.0,118.0
50%,57.0,127.0
75%,62.0,137.0
max,82.0,181.0


**Temuan:** ada 59 entri, 8 kategori, tanpa nilai kosong dan tanpa duplikat. Teksnya pendek: setiap jawaban hanya satu sampai dua kalimat. Karena itu satu FAQ cukup dijadikan satu unit retrieval dan tidak perlu di-*chunk*, sesuai ketentuan brief.

## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) dijadikan satu unit yang akan di-embed dan disimpan ke vector database. Question dan answer digabung menjadi satu teks per entri.

Alasan menggabungkan keduanya: pengguna kadang bertanya dengan kata yang ada di **jawaban**, bukan di pertanyaan FAQ. Contohnya "barang pecah" lebih dekat ke jawaban FAQ #16 ("foto/video kondisi paket") daripada ke pertanyaannya.

In [7]:
df["content"] = "Pertanyaan: " + df["question"] + "\nJawaban: " + df["answer"]
documents = df["content"].tolist()

print("Jumlah unit retrievable:", len(documents))
print("Contoh unit retrievable:\n")
print(documents[0])

Jumlah unit retrievable: 59
Contoh unit retrievable:

Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Setiap unit retrievable diubah menjadi vektor dengan **Gemini Embeddings API** (`gemini-embedding-2`).

Ada tiga keputusan teknis di sini:
1. **Format input asimetris.** `gemini-embedding-2` tidak lagi memakai parameter `task_type`. Sebagai gantinya, dokumentasi Google menyarankan menulis instruksi tugas di dalam teks:
   - dokumen: `title: {judul} | text: {isi}`
   - query: `task: search result | query: {pertanyaan}`
2. **Batching.** Beberapa dokumen dikirim dalam satu request. Hasilnya lebih cepat dan hemat kuota dibanding satu request per dokumen.
3. **Retry dengan backoff.** Kalau kena rate limit (HTTP 429), request otomatis diulang, sehingga notebook tidak berhenti di tengah jalan.

In [8]:
def _embed_batch(texts, max_retries=5):
    """Embed beberapa teks sekaligus. Setiap teks dibungkus Content sendiri supaya
    hasilnya satu vektor per teks (bukan satu vektor gabungan)."""
    contents = [types.Content(parts=[types.Part(text=t)]) for t in texts]
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=contents,
                config=types.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
            )
            return [e.values for e in result.embeddings]
        except Exception as err:  # rate limit / gangguan jaringan sementara
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"Embedding request failed ({err.__class__.__name__}), retry in {wait}s...")
            time.sleep(wait)


def format_document(row):
    return f"title: {row['category']} | text: {row['content']}"


def format_query(query):
    return f"task: search result | query: {query}"


def get_embeddings(texts, batch_size=20):
    vectors = []
    for start in range(0, len(texts), batch_size):
        vectors.extend(_embed_batch(texts[start:start + batch_size]))
    return np.array(vectors, dtype="float32")


def get_embedding(text):
    """Embedding untuk satu query pengguna."""
    return get_embeddings([format_query(text)])[0]


doc_inputs = [format_document(row) for _, row in df.iterrows()]
print("Contoh input embedding dokumen:\n", doc_inputs[0], "\n")

embeddings = get_embeddings(doc_inputs)
print("Shape embeddings:", embeddings.shape)

Contoh input embedding dokumen:
 title: Pemesanan | text: Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran. 

Shape embeddings: (59, 768)


In [9]:
# Verifikasi sebelum lanjut ke tahap penyimpanan
assert embeddings.shape == (len(df), EMBEDDING_DIM), "Jumlah/dimensi embedding tidak sesuai!"
assert not np.isnan(embeddings).any(), "Ada nilai NaN di embedding!"

norms = np.linalg.norm(embeddings, axis=1)
print(f"✓ {embeddings.shape[0]} vektor × {embeddings.shape[1]} dimensi")
print(f"Norma vektor: min={norms.min():.4f}, max={norms.max():.4f}")

✓ 59 vektor × 768 dimensi
Norma vektor: min=1.0000, max=1.0000


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

Indeks yang dipakai adalah **`IndexFlatIP`** (inner product) dengan vektor yang sudah dinormalisasi L2. Pada vektor ber-norma 1, nilai inner product sama dengan **cosine similarity**. Skor ini lebih mudah dibaca daripada L2 distance:

| Skor | Arti |
|---|---|
| 1.0 | Maknanya identik |
| ≈ 0 | Tidak berhubungan |
| Lebih tinggi | Lebih relevan |

`IndexFlat` melakukan pencarian *exact* (brute force). Untuk 59 vektor ini paling tepat. Indeks aproksimasi seperti IVF atau HNSW baru dibutuhkan kalau jumlahnya sudah ratusan ribu vektor ke atas.

In [10]:
faiss.normalize_L2(embeddings)          # normalisasi in-place → inner product = cosine similarity

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Jumlah vektor tersimpan di index:", index.ntotal)
assert index.ntotal == len(df), "Jumlah vektor di FAISS tidak sama dengan jumlah entri FAQ!"
print("✓ Jumlah vektor sesuai jumlah entri knowledge base")

Jumlah vektor tersimpan di index: 59
✓ Jumlah vektor sesuai jumlah entri knowledge base


In [11]:
# Simpan index ke disk + metadata, supaya bisa dipakai ulang tanpa embed ulang
INDEX_PATH = VECTOR_DIR / "faq_index.faiss"
META_PATH = VECTOR_DIR / "faq_index_metadata.json"

faiss.write_index(index, str(INDEX_PATH))
metadata = {
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": int(dimension),
    "index_type": "IndexFlatIP (L2-normalized → cosine similarity)",
    "n_vectors": int(index.ntotal),
    "faq_id_order": df["faq_id"].tolist(),   # posisi vektor ke-i = faq_id ke-i
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
META_PATH.write_text(json.dumps(metadata, indent=2))

# Verifikasi: index yang dimuat ulang identik
reloaded = faiss.read_index(str(INDEX_PATH))
assert reloaded.ntotal == index.ntotal
print(f"✓ Index tersimpan: {INDEX_PATH.relative_to(PROJECT_ROOT)} ({reloaded.ntotal} vektor)")
print(f"✓ Metadata tersimpan: {META_PATH.relative_to(PROJECT_ROOT)}")

✓ Index tersimpan: vector_store/faq_index.faiss (59 vektor)
✓ Metadata tersimpan: vector_store/faq_index_metadata.json


## 5. Mengimplementasikan Semantic Search

Fungsi ini mengambil beberapa entri FAQ yang paling relevan dengan query pengguna. Langkahnya:
1. Query diubah menjadi embedding dengan format query.
2. Embedding dinormalisasi.
3. FAISS mencari *top-k* entri dengan cosine similarity tertinggi.

In [12]:
def semantic_search(query, top_k=TOP_K):
    query_embedding = np.array([get_embedding(query)], dtype="float32")
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        row = df.iloc[idx]
        results.append({
            "rank": rank,
            "faq_id": int(row["faq_id"]),
            "category": row["category"],
            "question": row["question"],
            "answer": row["answer"],
            "score": round(float(score), 4),   # cosine similarity
        })
    return results


def show_results(query, results):
    print(f"Query: {query}")
    display(pd.DataFrame(results)[["rank", "faq_id", "category", "question", "score"]])


# Contoh percobaan
test_results = semantic_search("kenapa paket saya belum sampai?")
show_results("kenapa paket saya belum sampai?", test_results)

Query: kenapa paket saya belum sampai?


,rank,faq_id,category,question,score
0,1,12,Pengiriman,Bagaimana jika paket saya belum sampai melebih...,0.7836
1,2,14,Pengiriman,Kenapa status pengiriman saya tidak berubah se...,0.7462
2,3,52,Pengiriman,Kenapa nomor resi saya belum bisa dilacak di w...,0.7397


Query di atas tidak memakai kata "estimasi" atau "melebihi", tapi FAQ tentang paket terlambat tetap muncul di urutan atas. Inilah perbedaan semantic search dengan keyword search: yang dicocokkan adalah **makna**, bukan kata yang persis sama.

## 6. Mendesain Prompt dan Context

Prompt disusun dengan kerangka **Persona, Task, Context, Format**. Context diisi dari hasil semantic search, lengkap dengan ID FAQ dan skor relevansinya.

Ada tiga guardrail dalam prompt:
- **Grounding:** LLM hanya boleh menjawab dari context yang diberikan, bukan dari pengetahuan umumnya.
- **Fallback:** kalau context tidak memuat jawaban, LLM wajib memakai kalimat penolakan baku dan tidak boleh mengarang.
- **Jawaban parsial:** kalau topiknya ada tapi detailnya tidak, LLM menyampaikan info yang ada dan menyebut dengan jujur bagian yang tidak tersedia. Dengan begitu LLM tidak menolak berlebihan (*over-refusal*).

> **Iterasi prompt (v1 → v2).** Di prompt v1, aturan jawaban parsial dan aturan fallback ditulis sejajar. Hasilnya, query parsial #6 di section 8 kadang dijawab sebagian dan kadang ditolak penuh (1 dari 2 run). Di v2, prioritasnya dibuat eksplisit: fallback **hanya** dipakai kalau tidak ada satu pun FAQ yang membahas topik pertanyaan.
- **Sitasi:** jawaban menyebut ID FAQ yang dipakai, sehingga tim CS bisa memverifikasinya.

In [13]:
FALLBACK_MESSAGE = (
    "Maaf, saya belum memiliki informasi mengenai hal tersebut di FAQ kami. "
    "Silakan hubungi customer service kami untuk bantuan lebih lanjut."
)


def build_prompt(query, search_results):
    context = "\n\n".join(
        f"[FAQ #{r['faq_id']} | Kategori: {r['category']} | Skor relevansi: {r['score']:.3f}]\n"
        f"Pertanyaan: {r['question']}\n"
        f"Jawaban: {r['answer']}"
        for r in search_results
    )

    prompt = f"""### PERSONA
Kamu adalah asisten customer service sebuah platform e-commerce di Indonesia.
Kamu ramah, sopan, ringkas, dan selalu akurat.

### TASK
Jawab pertanyaan pelanggan HANYA berdasarkan informasi pada bagian CONTEXT.
Aturan:
1. Jangan gunakan pengetahuan umum atau asumsi di luar CONTEXT.
2. Tidak semua FAQ di CONTEXT pasti relevan, pilih yang benar-benar menjawab pertanyaan.
3. Jika ada FAQ yang membahas TOPIK yang sama tetapi tidak memuat detail spesifik yang ditanyakan,
   JANGAN gunakan kalimat fallback. Sampaikan informasi yang tersedia, lalu jelaskan dengan jujur
   bahwa detail tersebut tidak tersedia di FAQ dan sarankan menghubungi customer service.
4. Gunakan kalimat fallback HANYA jika tidak ada satu pun FAQ di CONTEXT yang membahas topik
   pertanyaan. Dalam kasus itu, balas PERSIS dengan kalimat ini:
   "{FALLBACK_MESSAGE}"

### CONTEXT
{context}

### FORMAT
- Bahasa Indonesia yang natural dan sopan, sapa pelanggan dengan "Anda".
- Maksimal 4 kalimat; gunakan langkah bernomor jika jawabannya berupa prosedur.
- Akhiri dengan baris "Sumber: FAQ #<id>" (pisahkan dengan koma jika lebih dari satu).
  Jangan tambahkan baris Sumber jika memakai kalimat fallback.

### PERTANYAAN PELANGGAN
{query}

### JAWABAN"""
    return prompt


# Lihat contoh prompt utuh yang dikirim ke LLM
print(build_prompt("kenapa paket saya belum sampai?", test_results))

### PERSONA
Kamu adalah asisten customer service sebuah platform e-commerce di Indonesia.
Kamu ramah, sopan, ringkas, dan selalu akurat.

### TASK
Jawab pertanyaan pelanggan HANYA berdasarkan informasi pada bagian CONTEXT.
Aturan:
1. Jangan gunakan pengetahuan umum atau asumsi di luar CONTEXT.
2. Tidak semua FAQ di CONTEXT pasti relevan, pilih yang benar-benar menjawab pertanyaan.
3. Jika ada FAQ yang membahas TOPIK yang sama tetapi tidak memuat detail spesifik yang ditanyakan,
   JANGAN gunakan kalimat fallback. Sampaikan informasi yang tersedia, lalu jelaskan dengan jujur
   bahwa detail tersebut tidak tersedia di FAQ dan sarankan menghubungi customer service.
4. Gunakan kalimat fallback HANYA jika tidak ada satu pun FAQ di CONTEXT yang membahas topik
   pertanyaan. Dalam kasus itu, balas PERSIS dengan kalimat ini:
   "Maaf, saya belum memiliki informasi mengenai hal tersebut di FAQ kami. Silakan hubungi customer service kami untuk bantuan lebih lanjut."

### CONTEXT
[FAQ #12 | Kateg

## 7. Mengintegrasikan LLM

Semantic search dan pemanggilan Gemini digabung menjadi satu alur: **query → retrieval → prompt → jawaban**.

Beberapa keputusan di sini:
- **`temperature` rendah (0.2).** Untuk asisten CS, konsistensi dan akurasi lebih penting daripada kreativitas.
- **Model `gemini-3.5-flash-lite`.** Tugas LLM di sini hanya merangkai jawaban dari context yang sudah diberikan, jadi model ringan sudah cukup. Biayanya lebih rendah dan latensinya lebih kecil.
- **Retry dan fallback.** Kalau model utama gagal (misalnya overload, timeout, atau kuota habis), request diulang beberapa kali. Kalau tetap gagal, sistem pindah ke `gemini-3.1-flash-lite`. Untuk kuota harian yang sudah habis (HTTP 429), retry tidak menolong, jadi langsung pindah model.

In [14]:
def call_llm(prompt, max_retries=3):
    """Panggil model utama dengan retry; jika tetap gagal, pindah ke model cadangan."""
    for model in (LLM_MODEL, LLM_FALLBACK_MODEL):
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=types.GenerateContentConfig(temperature=0.2),
                )
                return response.text.strip()
            except Exception as err:  # 429 rate limit, 503 overload, 504 timeout, dll.
                if "PerDay" in str(err):   # kuota harian habis → retry percuma, langsung ganti model
                    print(f"[{model}] daily quota exhausted.")
                    break
                wait = 2 ** attempt
                print(f"[{model}] request failed ({err.__class__.__name__}), retry in {wait}s...")
                time.sleep(wait)
        print(f"[{model}] still failing, switching to fallback model...")
    raise RuntimeError("All LLM models failed. Check your API key / quota / Gemini status.")


def generate_answer(query, top_k=TOP_K):
    search_results = semantic_search(query, top_k=top_k)   # 1. retrieve
    prompt = build_prompt(query, search_results)            # 2. augment (prompt + context)
    answer = call_llm(prompt)                               # 3. generate
    return answer, search_results


answer, sources = generate_answer("apakah bisa bayar pakai cicilan?")
print("Jawaban LLM:")
print(answer)
print()
print("Sumber FAQ yang digunakan:")
for s in sources:
    print(f"- FAQ #{s['faq_id']} (skor {s['score']:.3f}): {s['question']}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Jawaban LLM:
Halo! Ya, Anda bisa membayar dengan cicilan menggunakan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank. 

Sumber: FAQ #20

Sumber FAQ yang digunakan:
- FAQ #20 (skor 0.823): Apakah saya bisa membayar dengan cicilan?
- FAQ #22 (skor 0.733): Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- FAQ #17 (skor 0.730): Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Sistem diuji dengan **8 query** yang terbagi ke empat skenario:

| Tipe | Jumlah | Yang diuji |
|---|---|---|
| `direct` | 2 | Kalimat mirip dengan FAQ asli (*sanity check*) |
| `paraphrase` | 3 | Sinonim atau bahasa gaul, untuk menguji pemahaman makna dan bukan sekadar kata kunci |
| `partial` | 1 | Topiknya ada di FAQ, tapi detail yang ditanyakan tidak ada |
| `out_of_kb` | 2 | Tidak ada jawabannya di FAQ, termasuk satu yang topiknya mirip (pengiriman) sebagai jebakan |

Setiap query diberi `expected_faq_id` (FAQ yang seharusnya terambil) supaya retrieval bisa dinilai **secara objektif** dengan metrik **Hit@1** dan **Hit@3**, bukan hanya dengan penilaian visual.

In [15]:
test_queries = [
    # 1-2: jelas ada jawabannya di FAQ
    {"query": "Bagaimana cara membatalkan pesanan yang sudah dibuat?", "type": "direct", "expected_faq_id": 3},
    {"query": "Metode pembayaran apa saja yang tersedia?", "type": "direct", "expected_faq_id": 17},
    # 3-5: parafrase / sinonim / bahasa sehari-hari
    {"query": "Saldo saya sudah terpotong tapi pesanannya masih tertulis belum dibayar", "type": "paraphrase", "expected_faq_id": 21},
    {"query": "paketnya udah nyampe tapi barangnya pecah, harus gimana?", "type": "paraphrase", "expected_faq_id": 16},
    {"query": "aku lupa kata sandi buat masuk", "type": "paraphrase", "expected_faq_id": 32},
    # 6: topik ada, tapi detail yang ditanya tidak ada
    {"query": "Berapa minimal belanja supaya bisa pakai cicilan?", "type": "partial", "expected_faq_id": 20},
    # 7-8: tidak ada jawabannya di FAQ
    {"query": "Apakah bisa kirim barang ke luar negeri?", "type": "out_of_kb", "expected_faq_id": None},
    {"query": "Bagaimana cara mendaftar menjadi penjual dan membuka toko di platform ini?", "type": "out_of_kb", "expected_faq_id": None},
]

test_log = []
for i, t in enumerate(test_queries, start=1):
    answer, sources = generate_answer(t["query"])
    retrieved_ids = [s["faq_id"] for s in sources]
    exp = t["expected_faq_id"]
    test_log.append({
        "no": i,
        "type": t["type"],
        "query": t["query"],
        "expected_faq_id": exp if exp is not None else "-",
        "retrieved_top3": retrieved_ids,
        "top1_score": sources[0]["score"],
        "hit@1": (retrieved_ids[0] == exp) if exp is not None else None,
        "hit@3": (exp in retrieved_ids) if exp is not None else None,
        "fallback_used": FALLBACK_MESSAGE[:40] in answer,
        "answer": answer,
        "sources": sources,
    })

    print(f"[{i}] ({t['type']}) Query: {t['query']}")
    for s in sources:
        print(f"    #{s['rank']} FAQ {s['faq_id']:>2} | skor {s['score']:.3f} | {s['question']}")
    print("Jawaban:", answer)
    print("-" * 100)

[1] (direct) Query: Bagaimana cara membatalkan pesanan yang sudah dibuat?
    #1 FAQ  3 | skor 0.831 | Bagaimana cara membatalkan pesanan yang sudah dibuat?
    #2 FAQ  5 | skor 0.737 | Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?
    #3 FAQ  8 | skor 0.727 | Kenapa pesanan saya otomatis dibatalkan sistem?
Jawaban: Halo! Anda dapat membatalkan pesanan yang sudah dibuat melalui halaman 'Pesanan Saya' selama status pesanan masih 'Diproses'. Namun, perlu diingat bahwa setelah status berubah menjadi 'Dikirim', pembatalan sudah tidak dapat dilakukan. 

Sumber: FAQ #3
----------------------------------------------------------------------------------------------------
[2] (direct) Query: Metode pembayaran apa saja yang tersedia?
    #1 FAQ 17 | skor 0.819 | Metode pembayaran apa saja yang tersedia?
    #2 FAQ 22 | skor 0.752 | Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
    #3 FAQ 20 | skor 0.748 | Apakah saya bisa membayar dengan cicilan?
Jawaban

In [16]:
# Ringkasan hasil pengujian
summary = pd.DataFrame(test_log)[
    ["no", "type", "query", "expected_faq_id", "retrieved_top3", "top1_score", "hit@1", "hit@3", "fallback_used"]
]
display(summary)

in_kb = summary[summary["type"] != "out_of_kb"]
out_kb = summary[summary["type"] == "out_of_kb"]
print(f"Hit@1 (query yang ada di KB): {in_kb['hit@1'].mean():.0%}  ({int(in_kb['hit@1'].sum())}/{len(in_kb)})")
print(f"Hit@3 (query yang ada di KB): {in_kb['hit@3'].mean():.0%}  ({int(in_kb['hit@3'].sum())}/{len(in_kb)})")
print(f"Fallback dipakai pada query out-of-KB: {int(out_kb['fallback_used'].sum())}/{len(out_kb)}")
print(f"Fallback (salah) dipakai pada query in-KB: {int(in_kb['fallback_used'].sum())}/{len(in_kb)}")
print()
print(f"Skor top-1 terendah query in-KB  : {in_kb['top1_score'].min():.3f}")
print(f"Skor top-1 tertinggi query out-KB: {out_kb['top1_score'].max():.3f}")

,no,type,query,expected_faq_id,retrieved_top3,top1_score,hit@1,hit@3,fallback_used
0,1,direct,Bagaimana cara membatalkan pesanan yang sudah ...,3,"[3, 5, 8]",0.8309,True,True,False
1,2,direct,Metode pembayaran apa saja yang tersedia?,17,"[17, 22, 20]",0.8190,True,True,False
2,3,paraphrase,Saldo saya sudah terpotong tapi pesanannya mas...,21,"[21, 19, 8]",0.8265,True,True,False
3,4,paraphrase,"paketnya udah nyampe tapi barangnya pecah, har...",16,"[16, 12, 54]",0.7753,True,True,False
4,5,paraphrase,aku lupa kata sandi buat masuk,32,"[32, 55, 35]",0.7770,True,True,False
5,6,partial,Berapa minimal belanja supaya bisa pakai cicilan?,20,"[20, 17, 18]",0.7912,True,True,False
6,7,out_of_kb,Apakah bisa kirim barang ke luar negeri?,-,"[15, 11, 10]",0.6784,None,None,True
7,8,out_of_kb,Bagaimana cara mendaftar menjadi penjual dan m...,-,"[31, 1, 50]",0.7122,None,None,True


Hit@1 (query yang ada di KB): 100%  (6/6)
Hit@3 (query yang ada di KB): 100%  (6/6)
Fallback dipakai pada query out-of-KB: 2/2
Fallback (salah) dipakai pada query in-KB: 0/6

Skor top-1 terendah query in-KB  : 0.775
Skor top-1 tertinggi query out-KB: 0.712


**Uji konsistensi.** Output LLM bersifat non-deterministik, sehingga satu kali run belum cukup untuk menyimpulkan apa-apa. Dua query yang paling rawan, yaitu `partial` (#6) dan `out_of_kb` yang paling mirip FAQ (#8), masing-masing dijalankan 3 kali untuk melihat apakah keputusan LLM (menjawab atau fallback) stabil.

In [18]:
consistency = []
for t in [test_queries[5], test_queries[7]]:
    outcomes = []
    for _ in range(3):
        ans, _ = generate_answer(t["query"])
        outcomes.append("fallback" if FALLBACK_MESSAGE[:40] in ans else "menjawab")
    expected = "fallback" if t["type"] == "out_of_kb" else "menjawab"
    consistency.append({"type": t["type"], "query": t["query"], "expected": expected,
                        "outcomes (3x)": outcomes, "konsisten & benar": all(o == expected for o in outcomes)})
pd.DataFrame(consistency)

[gemini-3.5-flash-lite] request failed (ServerError), retry in 1s...
[gemini-3.5-flash-lite] request failed (ServerError), retry in 1s...


,type,query,expected,outcomes (3x),konsisten & benar
0,partial,Berapa minimal belanja supaya bisa pakai cicilan?,menjawab,"[menjawab, menjawab, menjawab]",True
1,out_of_kb,Bagaimana cara mendaftar menjadi penjual dan m...,fallback,"[fallback, fallback, fallback]",True


In [19]:
# Simpan hasil mentah pengujian ke documentation/ (analisis ada di README.md & cell di bawah)
lines = [
    "# Retrieval Test Results",
    "",
    f"- Embedding model: `{EMBEDDING_MODEL}` ({EMBEDDING_DIM} dim)",
    f"- LLM: `{LLM_MODEL}` (fallback `{LLM_FALLBACK_MODEL}`, temperature 0.2)",
    f"- Top-k: {TOP_K}, similarity: cosine (FAISS IndexFlatIP)",
    f"- Dijalankan: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}",
    "",
    "> File ini dibuat otomatis oleh notebook (section 8). Analisis relevansi ada di README.md.",
    "",
    "## Ringkasan",
    "",
    "| No | Tipe | Query | Expected | Top-3 retrieved (skor) | Hit@1 | Hit@3 | Fallback |",
    "|---|---|---|---|---|---|---|---|",
]
fmt = lambda v: "-" if v is None else ("✅" if v else "❌")
for r in test_log:
    top3 = ", ".join(f"#{s['faq_id']} ({s['score']:.3f})" for s in r["sources"])
    lines.append(f"| {r['no']} | {r['type']} | {r['query']} | {r['expected_faq_id']} | {top3} | "
                 f"{fmt(r['hit@1'])} | {fmt(r['hit@3'])} | {'ya' if r['fallback_used'] else 'tidak'} |")
lines += ["", "## Uji konsistensi (3x run)", "", "| Tipe | Query | Expected | Hasil 3x | Konsisten & benar |", "|---|---|---|---|---|"]
for c in consistency:
    lines.append(f"| {c['type']} | {c['query']} | {c['expected']} | {', '.join(c['outcomes (3x)'])} | "
                 f"{'✅' if c['konsisten & benar'] else '❌'} |")
lines += ["", "## Jawaban LLM per query", ""]
for r in test_log:
    lines += [f"### {r['no']}. ({r['type']}) {r['query']}", ""]
    lines += ["> " + line if line else ">" for line in r["answer"].splitlines()]
    lines.append("")

out_path = DOC_DIR / "retrieval_test_results.md"
out_path.write_text("\n".join(lines), encoding="utf-8")
print("✓ Hasil pengujian tersimpan di", out_path.relative_to(PROJECT_ROOT))

✓ Hasil pengujian tersimpan di documentation/retrieval_test_results.md


### Catatan Analisis Relevansi

*Angka di bawah berasal dari run di atas. Retrieval bersifat deterministik, sedangkan redaksi jawaban LLM bisa sedikit berbeda antar-run.*

**1. Retrieval: 6/6 query in-KB tepat di peringkat 1 (Hit@1 = Hit@3 = 100%).**
- Query *direct* (#1, #2) mendapat skor tertinggi (0.83 dan 0.82), wajar karena kalimatnya hampir identik dengan FAQ.
- Query *parafrase* (#3–#5) tetap tepat walaupun hampir tidak ada kata yang sama: "saldo terpotong" → "pembayaran dipotong", "pecah" → "kondisi rusak", "kata sandi … masuk" → "password … login". Ini bukti semantic search menangkap **makna**. Keyword search kemungkinan besar gagal di #4 dan #5.
- Harganya: semakin informal bahasanya, skor dan margin keyakinannya turun. Pada #5, selisih peringkat 1 dan 2 hanya **0.059** (0.777 vs 0.718, yaitu FAQ "tidak bisa login").

**2. Skor cosine bukan probabilitas.** Query di luar FAQ tetap mendapat skor 0.68–0.71, tidak jauh dari skor minimum query in-KB (0.775). FAISS selalu mengembalikan top-3 meskipun tidak ada yang relevan. Celah 0.063 di sampel ini menunjukkan threshold sekitar 0.74 *mungkin* bisa dipakai, tapi 8 query belum cukup untuk mengkalibrasinya.

**3. Query #8 berhasil "menipu" retriever.** "Mendaftar menjadi **penjual**" mengambil FAQ #31 "mendaftar **akun baru**" (0.712) karena kemiripan kata. Secara makna, retrieval-nya salah, tapi LLM tetap menolak dengan benar. Di sini guardrail prompt berfungsi sebagai **lapisan pertahanan kedua**. Tanpa guardrail itu, LLM bisa menjawab cara daftar akun pembeli seolah-olah cara menjadi penjual (halusinasi yang *terlihat* valid).

**4. Query #7 (pengiriman ke luar negeri).** Retriever mengambil 3 FAQ pengiriman yang topiknya berdekatan, dengan skor top-1 terendah dari semua query (0.678). LLM tidak mengarang informasi pengiriman internasional.

**5. Iterasi prompt v1 → v2 (temuan terpenting).** Dengan prompt v1, query parsial #6 dijawab sebagian di run pertama tetapi **ditolak penuh di run kedua** (*over-refusal*), padahal FAQ #20 memuat info berguna. Prompt v2 membuat prioritasnya eksplisit: fallback *hanya* dipakai kalau tidak ada FAQ yang membahas topiknya. Setelah itu, uji konsistensi 3x menunjukkan #6 selalu menjawab sebagian dan #8 selalu fallback. Pelajarannya: **satu run tidak cukup untuk menilai LLM** karena output-nya non-deterministik.

**6. Kualitas jawaban.** Semua jawaban in-KB faktual sesuai FAQ, menyertakan `Sumber: FAQ #id`, maksimal 4 kalimat, dan memakai langkah bernomor untuk prosedur. LLM juga mengabaikan FAQ pengganggu di context. Catatan minor: di #1 LLM memaksakan format langkah bernomor untuk info yang cukup satu kalimat.

Keterbatasan sistem dan rencana pengembangan dibahas lengkap di `README.md` (section 6).